In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Subtask 01 - Data Exploration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri

## 1. Loading Data

In [ ]:
def load_data(data_path):
    data = np.load(data_path, allow_pickle=True).item()
    return data

data_path = "/kaggle/input/elec-70127-fno/data/train/data_02005.npy"

result = load_data(data_path)

for key, item in result.items():
    print(key, item.shape)

## 2. Data Visulisation

In [ ]:
def visulise_data(data_path, cmap="jet"):
    zeta = 10

    data = load_data(data_path)

    coord = data.get("coord")
    face = data.get("cell_node_list")
    value = data.get("field_node")[:, 2] # here we are visulising the pressure field

    left, right = min(coord[:, 0]), max(coord[:, 0])
    bottom, top = min(coord[:, 1]), max(coord[:, 1])

    width = zeta * (right - left)
    height = zeta * (top - bottom)

    triangle = tri.Triangulation(coord[:, 0], coord[:, 1], face)

    fig, ax = plt.subplots(figsize=(width, height))

    ax.set_aspect("equal")
    ax.triplot(triangle, 'ko-', ms=0.1, lw=0.1)
    ax.tripcolor(triangle, value, cmap=cmap, alpha=0.4)

    return fig, ax

In [ ]:
visulise_data(data_path)

## 3. Data statistics

In [ ]:
import glob
from natsort import natsorted

In [ ]:
# Make it a task
def get_mean_std(data_folder_path):
    data_list = []

    file_path = os.path.join(data_folder_path, "data_*.npy")
    file_names = glob.glob(file_path)
    file_names = natsorted(file_names)

    for file_name in file_names:
        data = load_data(file_name)
        data_list.append(data.get("field_node"))

    data = np.concatenate(data_list, axis=0)

    mean = np.mean(data, axis=0, keepdims=True)
    std = np.std(data, axis=0, keepdims=True)

    return mean, std


In [ ]:
mean, std = get_mean_std(f"/kaggle/input/elec-70127-fno/data/train/")

In [ ]:
from scipy.interpolate import griddata


def get_grid_and_mask(data):

    r = 0.05
    x_center = 0.3
    y_center = 0.2

    grid_num_x = 320
    grid_num_y = 80

    coord = data.get("coord")
    field_node = data.get("field_node")

    x_min, x_max = min(coord[:, 0]), max(coord[:, 0])
    y_min, y_max = min(coord[:, 1]), max(coord[:, 1])

    x_coord, y_coord = np.meshgrid(
        np.linspace(x_min, x_max, grid_num_x),
        np.linspace(y_min, y_max, grid_num_y)
    )

    x_coord, y_coord = x_coord.flatten(), y_coord.flatten()

    # calculate mask
    dis_from_center = np.sqrt((x_coord - x_center)**2 + (y_coord - y_center)**2)
    mask = dis_from_center <= r

    grid_data = []
    num_dim = field_node.shape[1]
    for i in range(num_dim):
        grid_data_i = griddata(
            coord, field_node[:, i], (x_coord, y_coord), method='cubic'
        )

        grid_data_i[mask] = 0
        grid_data_i = grid_data_i.reshape(grid_num_y, grid_num_x)
        grid_data.append(grid_data_i)

    grid_data = np.stack(grid_data, axis=0)

    mask_grid = mask.reshape(1, grid_num_y, grid_num_x)

    return grid_data, mask_grid

In [ ]:
from tqdm import tqdm

def post_process(dataset_dir, save_dir=None):
    file_names = glob.glob(os.path.join(dataset_dir, "data_*.npy"))
    file_names = natsorted(file_names)

    for file_name in tqdm(file_names, desc="Processing files"):
        data = load_data(file_name)
        grid_data, mask = get_grid_and_mask(data)
        data["field_conv"] = grid_data
        data["mask"] = mask

        if save_dir is None:
            # overwrite original file
            np.save(file_name, data)
        else:
            # 1. create parent folder if not exist (recursively)
            os.makedirs(save_dir, exist_ok=True)

            # 2. keep original filename
            base_name = os.path.basename(file_name)
            file_name_new = os.path.join(save_dir, base_name)

            np.save(file_name_new, data)

In [ ]:
post_process(dataset_dir = "/kaggle/input/elec-70127-fno/data/train/", save_dir="/kaggle/working/train")

In [ ]:
post_process(dataset_dir = "/kaggle/input/elec-70127-fno/data/test/", save_dir="/kaggle/working/test")

In [ ]:
post_process(dataset_dir = "/kaggle/input/elec-70127-fno/data/val_task3/", save_dir="/kaggle/working/val_task3")

In [ ]:
data_root = "/kaggle/working/"

In [ ]:
data = load_data(data_root + "train/data_02005.npy")

grid_data = data["field_conv"]
print(grid_data.shape)
plt.imshow(grid_data[0], cmap="jet")
plt.show()

# Subtask 02 - Dataset construction and Data Loading

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from natsort import natsorted
from scipy.interpolate import griddata

In [ ]:
import glob
class CNNDataset(Dataset):
    """
    A custom PyTorch dataset for loading spatiotemporal data in a format suitable for CNN-based models.

    This dataset handles time-series data stored in `.npy` files,
    The dataset loads consecutive time frames, applies transformations,
    and formats the data for training models.

    Args:
        dataset_dir (str, optional): Path to the directory containing `.npy` dataset files.
        t_dim (int, optional): The number of time steps to include per sample. Default is 10.
        transform (callable, optional): A function to apply transformations to input data.
        target_transform (callable, optional): A function to apply transformations to target data.

    Attributes:
        file_names (list): A sorted list of file paths to dataset files.
        t_dim (int): The number of time steps per sample.
        transform (callable): Transformation function for input data.
        target_transform (callable): Transformation function for target data.

    Task:
    - Implement `__len__`:
        - Compute the number of samples by dividing dataset length by `(2 * t_dim)`.
    - Implement `__getitem__`:
        - Compute index ranges for `in_data` and `out_data`.
        - Load and process the input-output data pairs.
        - Apply transformations if provided.
        - Stack time-series data along the temporal axis.
        - Ensure the final shape follows `(x1, x2, t, channel)` format. where x1 and x2 are spatial dimensions and t is the number of time steps.

    Note:
    - This dataset assumes `.npy` files contain dictionary-like data.
    - Set `t_dim == 10` based on the sequence length training.
    - Feel free to modify the preprocessing pipeline to fit specific model architectures - if that is necessary.
    """

    def __init__(self, dataset_dir=None, t_dim=10, transform=None, target_transform=None):
        self.t_dim=t_dim
        self.transform = transform
        self.target_transform = target_transform

        self.file_names = glob.glob(os.path.join(dataset_dir, "data_*.npy"))
        self.file_names = natsorted(self.file_names)

    def __len__(self):
      # Your code here
        return

    def load_data(self, data_path):
        data = np.load(data_path, allow_pickle=True).item()
        return data

    def __getitem__(self, idx):
        # Your code here
        return in_data, out_data

In [ ]:
dataset_dir = f"{data_root}/test/"
print(dataset_dir)

dataset = CNNDataset(dataset_dir)
print(len(dataset))

idx = 20
sample, target = dataset[idx]
print(sample.shape, target.shape)

## 2. Data Transformation

In [ ]:
class Normalise:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, data):
        data = (data - self.mean) / self.std
        return data


class ToFloatTransform:
    def __init__(self, scale=True):
        return

    def __call__(self, data):
        data = torch.from_numpy(data).float()
        return data


class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, data):
        for transform in self.transforms:
            data = transform(data)
        return data

In [ ]:
transform = Compose([
    Normalise(mean = mean.reshape(-1, 1, 1),
              std = std.reshape(-1, 1, 1)),
    ToFloatTransform(),
])

dataset = CNNDataset(dataset_dir = dataset_dir, transform=transform, target_transform=transform)

idx = 0
data_in, target = dataset[idx]

conv_in = data_in
print(conv_in.shape)

## 3. Data Loader

In [ ]:
Loader = DataLoader(dataset=dataset, num_workers=4, batch_size=16)

# Subtask 03 - FNO

![](https://www.googleapis.com/download/storage/v1/b/kaggle-user-content/o/inbox%2F24644801%2Fee12956a1e03360c33eacf463cc67094%2FWX20260202-0053412x.png?generation=1769993637576752&alt=media)

## Understanding `SpectralConv3d` Class

1. In the `class SpectralConv3d(nn.Module)` class, why we use `self.weights1, self.weights2 ...`

2. Why there is a 'self.scale = (1 / (in_channels * out_channels))` in the code? what is the purpose of this line?

3. What is the line `torch.einsum("bixyz,ioxyz->boxyz", input, weights)` doing here? Could you explain it? Could you also give a pytorch implementation with out einsum doing the same thing (porvide demo in the code cell below)? Why we want to use einsum?

4. Try to go through the `forward` methods, annotate this methods line by line explaining the purpose of each line.

## Your Answers here:
1.
...

In [ ]:
# Your Code here

from torch import nn
import torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SpectralConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2, modes3):
        super(SpectralConv3d, self).__init__()
        """
        3D Fourier layer. It does FFT, linear transform, and Inverse FFT.
        """
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1 # Number of Fourier modes to multiply, at most floor(N/2) + 1
        self.modes2 = modes2
        self.modes3 = modes3

        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))
        self.weights3 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))
        self.weights4 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.cfloat))

    # Complex multiplication
    def compl_mul3d(self, input, weights):
        # (batch, in_channel, x,y,t ), (in_channel, out_channel, x,y,t) -> (batch, out_channel, x,y,t)
        return torch.einsum("bixyz,ioxyz->boxyz", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[-3,-2,-1])

        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-3), x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, :self.modes1, :self.modes2, :self.modes3], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, -self.modes1:, :self.modes2, :self.modes3], self.weights2)
        out_ft[:, :, :self.modes1, -self.modes2:, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, :self.modes1, -self.modes2:, :self.modes3], self.weights3)
        out_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3] = \
            self.compl_mul3d(x_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3], self.weights4)

        x = torch.fft.irfftn(out_ft, s=(x.size(-3), x.size(-2), x.size(-1)))
        return x



class FNO3d(nn.Module):
    def __init__(self,
                 modes1=8,
                 modes2=8,
                 modes3=8,
                 width=36
                ):
        super(FNO3d, self).__init__()
        """
        3D Fourier Neural Operator (FNO) for solving PDEs in spatiotemporal domains.

        This model consists of three Fourier layers followed by point-wise feed-forward
        layers to learn complex spatial-temporal dependencies.

        Args:
            modes1 (int, optional): Number of Fourier modes to keep in the first dimension. Default is 8.
            modes2 (int, optional): Number of Fourier modes to keep in the second dimension. Default is 8.
            modes3 (int, optional): Number of Fourier modes to keep in the third (temporal) dimension. Default is 5.
            width (int, optional): Number of feature channels in Fourier and convolution layers. Default is 36.

        Attributes:
            fc0 (nn.Linear): Initial linear layer to project input channels to `width`.
            conv0, conv1, conv2 (SpectralConv3d): Spectral convolution layers for learning frequency representations.
            w0, w1, w2 (nn.Conv1d): Pointwise convolution layers for additional processing.
            fc1, fc2 (nn.Linear): Fully connected layers for post-processing.

        Task:
        - Implement the `forward` method:
            - Apply `fc0` to project input to `width` channels.
            - Use `permute` to rearrange dimensions for spectral convolutions.
            - Apply `conv0`, `conv1`, `conv2` sequentially, with `relu` activations.
            - Apply `w0`, `w1`, `w2` to perform pointwise transformations.
            - Pass through `fc1` and `fc2` to obtain final outputs.
        """
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3
        self.width = width
        """
        3 channels for velocity_x, velocity_y, pressure
        """
        self.fc0 = nn.Linear(3, self.width)
        self.conv0 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv1 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv2 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)

        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        batchsize = x.shape[0]
        size_x, size_y, size_z = x.shape[1], x.shape[2], x.shape[3]
        # print(x.shape)
        x = self.fc0(x)
        x = x.permute(0, 4, 1, 2, 3)

        # Your Code Here

        x = x.permute(0, 2, 3, 4, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)

        return x

In [ ]:
model = FNO3d(10, 10, 5)
dummy = torch.randn((8, 90, 320, 10, 3))
print(model(dummy).shape)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    avg_loss = 0.0
    for batch in loader:
        in_data, out_data = batch
        in_data, out_data = in_data.to(device), out_data.to(device)
        optimizer.zero_grad()
        pred = model(in_data)
        loss = criterion(pred, out_data)
        avg_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_loss /= len(loader)
    return loss, model


def test_one_epoch(model, loader, criterion):
    model.eval()
    avg_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            in_data, out_data = batch
            in_data, out_data = in_data.to(device), out_data.to(device)
            pred = model(in_data)
            loss = criterion(pred, out_data)
            avg_loss += loss.item()
    avg_loss /= len(loader)
    return loss




def train(epoch, model, optimizer, loader_train, loader_test):
    criterion = torch.nn.MSELoss()
    model = model.to(device)
    for i in range(epoch):
        loss, model = train_one_epoch(
            model=model, loader=loader_train, optimizer=optimizer, criterion=criterion)
        print(f"Epoch \t{i+1}, Train Loss \t{loss}")
        if (i+1) % 10 == 0:
            info = test_one_epoch(
                model=model, loader=loader_test, criterion=criterion)
            print(f"\t Epoch \t{i+1}, Test Loss \t{loss}")
            model_path = os.path.join(
                os.path.abspath("."),
                f"epoch_{i+1}.pth"
            )
            torch.save(model.state_dict(), model_path)
            print(model_path, "saved")


## Model Training and Tuning

Key Considerations:

What is the role of modes1, modes2, and modes3?
> These parameters define the number of Fourier modes retained in each spatial dimension, controlling the spectral resolution of the model.

What loss function did you use to train the model?
> Specify the loss function(s) experimented with and justify the choice in terms of stability, convergence, and accuracy.

Model Tuning and Optimization Log:

Provide a concise summary of your model tuning process, including:
* Alterations made to the baseline model to improve performance.
* Hyperparameter adjustments and their impact.
* The most effective modification that led to the best overall results.

Clearly document your reasoning behind each change to demonstrate an iterative, data-driven approach to model optimization.

In [ ]:
epoch = 20
lr = 1e-3
bs = 6

mode = 8
model = FNO3d(modes1 = mode, modes2=mode, modes3=4, width=36)

train_set = CNNDataset(
    dataset_dir = f"{data_root}/train/",
    transform=transform, target_transform=transform)
test_set = CNNDataset(
    dataset_dir = f"{data_root}/test/",
    transform=transform, target_transform=transform)

loader_train = DataLoader(train_set, batch_size=bs, num_workers=4)
loader_test = DataLoader(test_set, batch_size=bs, num_workers=4)

# Init optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=lr,
    weight_decay=1e-6,
)

train(epoch=epoch, model=model, optimizer=optimizer, loader_train=loader_train, loader_test=loader_test)

In [ ]:
def load_model(model, weight_path, strict=False):
    """
    Loads pre-trained weights into a PyTorch model from a given file path.

    Args:
        model (torch.nn.Module): The PyTorch model.
        weight_path (str): File path to the pre-trained model weights.

    Returns:
        torch.nn.Module: The model with loaded weights.
    """
    try:
        model.load_state_dict(torch.load(weight_path), strict=strict)
    except RuntimeError:
        state_dict = torch.load(weight_path, map_location=device)
        model.load_state_dict(state_dict, strict=strict)
    return model


def denormalise(data, mean, std):
    return data * std + mean

def normalise(data, mean, std):
    return (data - mean) / std

def calc_norm_error(pred, truth, ord=2):
    """
    Compute the relative error of two input. input are supposed to have
    shape of (channel, height, width). Other shape might work but please
    be careful and aware of what you are doing.

    pred: The prediction output by model.
    truth: The ground truth against which error are measured.
    ord: the order of norm.
    """
    pred, truth = [vec.flatten() for vec in [pred, truth]]
    nume = torch.linalg.norm((pred - truth), ord=ord)
    deno = torch.linalg.norm(truth, ord=ord)
    return nume / deno

def calculate_error_and_inference(dataloader, model, mean, std):
    model.to(device)
    model.eval()
    error = 0.0
    pred_list = []
    ground_list = []
    with torch.no_grad():
        for batch in dataloader:
            data_in, target = batch
            data_in, target = data_in, target
            data_in, target = data_in.to(device), target.to(device)
            pred = model(data_in)
            pred, target = denormalise(pred, mean, std), denormalise(target, mean, std)
            pred_list.append(pred.detach().cpu().numpy())
            ground_list.append(target.detach().cpu().numpy())
            error_i = calc_norm_error(pred, target)
            error += error_i.detach()
    error /= (len(dataloader) * dataloader.batch_size)
    return error, pred_list, ground_list


In [ ]:
model = FNO3d(modes1 = mode, modes2=mode, modes3=4, width=36)

model = load_model(model, "./epoch_20.pth")
dataset_test = CNNDataset(
    dataset_dir = f"{data_root}/test/",
    transform=transform, target_transform=transform)
test_loader = DataLoader(dataset_test, batch_size=1, num_workers=8)

error, pred, ground = calculate_error_and_inference(
    test_loader, model,
    mean = torch.tensor(mean.reshape(1, 1, 1, 1, -1)).to(device),
    std = torch.tensor(std.reshape(1, 1, 1, 1, -1)).to(device),
)
print(error)

In [ ]:
def pred_to_df(data):
    reshaped = data.reshape(-1, 3)
    df = pd.DataFrame(reshaped, columns=['Feature_1', 'Feature_2', 'Feature_3'])
    df.index.name = "ROW_ID"
    return df

In [ ]:
file_dir = f"{data_root}/val_task3/"
file_names = glob.glob(os.path.join(file_dir, "data_*.npy"))
file_names = natsorted(file_names)

in_list = []
for i in range(len(file_names)):
    in_data = load_data(file_names[i]).get("field_conv")
    in_data = normalise(
        in_data,
        mean = mean.reshape(-1, 1, 1),
        std = std.reshape(-1, 1, 1),
    )
    in_list.append(in_data)

in_data = np.stack(in_list, axis=0)
in_data = in_data.transpose(2, 3, 0, 1)
in_data = torch.tensor(in_data).unsqueeze(0).float().to(device)
print(in_data.shape)

model.to(device)
model.eval()
with torch.no_grad():
    pred = model(in_data).detach().cpu().numpy()
    pred = denormalise(pred, mean, std)
print(pred.shape)

In [ ]:
pred = np.stack(pred, axis=0)
print(pred.shape)

In [ ]:
import pandas as pd
print(pred.shape)
# save file for submission 1
pred_1 = np.stack(pred, axis=0)
np.save("pred_1.npy", pred_1)
pred_to_df(pred_1).to_csv("/kaggle/working/submission.csv", index=True)

In [ ]:
import matplotlib.tri as tri
cmap = "jet"
zeta = 10

coord = data["coord"]
face = data["cell_node_list"]

left, right = min(coord[:, 0]), max(coord[:, 0])
bottom, top = min(coord[:, 1]), max(coord[:, 1])
idx = 9

value = pred[0, :, :, idx, 0]
plt.imshow(value)

## Generative AI Usage Log

Document your usage of generative AI.

### Case 1:

- **Tool Used:**  

- **Question / Context:**  

- **Prompt / Process:**  

- **How You Used the Output:**

### Case 2:

...


